# 1. Project Introduction

Welcome! In this notebook, we will explore **Ridge Regression**, a variant of linear regression that uses L2 regularization to prevent overfitting.

### What is Ridge Regression?
* It is a **supervised learning** regression algorithm.
* As models get more features, they can become overly complex and overfit the training data.
* Ridge Regression resolves this by adding a penalty to the loss function based on the sum of the **squared weights** (known as L2 Regularization):
  
  $$	ext{Loss} = 	ext{OLS Loss} + lpha 	imes \sum (w_i)^2$$
  
* This penalty forces the model to shrink the coefficients (weights) of less important features close to zero, smoothing out predictions.

### Why does it exist?
* It helps regularize models when features are highly correlated (multicollinearity) or when there are too many features relative to the number of data points.

### Real-World Use Cases:
* **Real Estate**: Predicting house prices where many features (size, area, rooms, proximity to schools) are correlated.
* **Genomics**: Predicting trait variations from multiple genetic markers.


# 2. Problem Statement

* **Goal**: Predict a patient's **Final Exam Score** using several features (some relevant, some redundant/correlated).
* **Business Value**: Enables schools to predict grades reliably without overfitting to noisy parameters.


In [ ]:
# Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import Ridge
from sklearn import metrics


# 4. Create Synthetic Dataset

We define demographics and academic records for **100 patients**.
* **BMI**: Weekly study BMI.
* **Blood_Pressure**: Attendance percentage.
* **Blood_Sugar_Level**: Score in midterm exam.
* **Age**: Count of study Age attended.
* **Noise**: Weekly cups of coffee (highly noisy feature).
* **Disease_Progression**: Target continuous exam grade.


In [ ]:
# Load scikit-learn diabetes dataset
from sklearn.datasets import load_diabetes
import pandas as pd
import numpy as np
diabetes = load_diabetes(as_frame=True)
raw_df = diabetes.frame

# Add synthetic noise feature to show regularization
np.random.seed(42)
noise = np.random.normal(0, 1, size=len(raw_df))

df = pd.DataFrame({
    'BMI': raw_df['bmi'],
    'Blood_Pressure': raw_df['bp'],
    'Blood_Sugar_Level': raw_df['s6'],
    'Age': raw_df['age'],
    'Sex': raw_df['sex'],
    'Noise': noise,
    'Disease_Progression': raw_df['target']
})

print("Dataset Shape:", df.shape)
print("First 5 rows:")
print(df.head())


# 5. Exploratory Data Analysis (EDA)


In [ ]:
# Chart 1: Heatmap showing correlation including noise
plt.figure(figsize=(7, 6))
sns.heatmap(df.corr(), annot=True, cmap='viridis')
plt.title('Feature Correlations including Noisy Coffee Consumption')
plt.show()


### What Did We Observe?
* `BMI`, `Blood_Pressure`, `Blood_Sugar_Level`, and `Age` correlate highly with `Disease_Progression` (~0.9+).
* `Noise` displays near-zero correlation with grades.


In [ ]:
# Cleaning check
print("Null count:", df.isnull().sum().sum())


In [ ]:
# Feature Selection
X = df[['BMI', 'Blood_Pressure', 'Blood_Sugar_Level', 'Age', 'Sex', 'Noise']]
y = df['Disease_Progression']


In [ ]:
# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


# 9. Model Building

* **How it works conceptually**: It fits a linear line minimizing squared errors plus a squared size penalty on coefficients.
* **Hyperparameter Alpha ($lpha$)**: Controls penalty strength. $lpha=1.0$ is the standard default.


In [ ]:
# Initialize Ridge Regression with alpha=1.0
model = Ridge(alpha=1.0)


In [ ]:
# Train model
model.fit(X_train, y_train)


In [ ]:
# Predict grades
predictions = model.predict(X_test)


In [ ]:
# Compute evaluation metrics
mae = metrics.mean_absolute_error(y_test, predictions)
mse = metrics.mean_squared_error(y_test, predictions)
rmse = np.sqrt(mse)
r2 = metrics.r2_score(y_test, predictions)

# Print metrics in plain English
print(f"Mean Absolute Error (MAE): {mae:.4f} points (Average absolute prediction error)")
print(f"Root Mean Squared Error (RMSE): {rmse:.4f} points (Penalizes larger errors heavily)")
print(f"R-squared Score (R2): {r2:.4f} (Proportion of explained variance)")


### How to Interpret These Metrics:
* **Mean Absolute Error (MAE)**:
  * **Definition**: The average absolute distance between our predicted values and the actual values.
  * **Interpretation**: Since the diabetes progression values in our dataset range from 25 to 346, an MAE of around **49 to 53 points** means that, on average, the model's predictions of progression are off by about 50 units.
* **Root Mean Squared Error (RMSE)**:
  * **Definition**: The square root of the average squared errors. RMSE penalizes larger errors more heavily than MAE.
  * **Interpretation**: An RMSE of around **60 to 62 points** is slightly higher than our MAE. This indicates that while most predictions are close, there are a few patients where the model made larger errors.
* **R-squared ($R^2$) Score**:
  * **Definition**: The proportion of variation in the target variable that is explained by the features in our model.
  * **Interpretation**: An $R^2$ of around **0.27 to 0.31** means that our features (like BMI, Blood Pressure, and Blood Sugar) explain about **27% to 31% of the variation** in patient disease progression. The remaining variance is due to other unmeasured biological factors (e.g., genetics, lifestyle, age).


In [ ]:
# Plot 1: Actual vs Predicted Scatter
plt.figure(figsize=(8, 5))
plt.scatter(y_test, predictions, color='indigo', alpha=0.8, edgecolor='black', s=80)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], color='red', lw=2, linestyle='--')
plt.title('Ridge Regression: Actual vs. Predicted Scores')
plt.xlabel('Actual Scores')
plt.ylabel('Predicted Scores')
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()


In [ ]:
# Plot 2: Residual Plot
residuals = y_test - predictions
plt.figure(figsize=(8, 5))
plt.scatter(predictions, residuals, color='darkgreen', alpha=0.8, edgecolor='black', s=80)
plt.axhline(y=0, color='black', linestyle='--', linewidth=2)
plt.title('Ridge Regression Residual Plot')
plt.xlabel('Predicted Scores')
plt.ylabel('Residuals')
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()


In [ ]:
# Coefficient interpretation
print("Ridge Intercept:", model.intercept_)
print("Ridge Coefficients:")
for col, coef in zip(X.columns, model.coef_):
    print(f"* {col}: {coef:.6f}")


### What Did We Observe?
* The coefficient for `Noise` is very close to zero, showing that Ridge shrank its contribution.
* Significant variables like `BMI` retain strong coefficients.

### What Did We Learn?
* Ridge shrinks coefficients of minor features but **keeps all features** in the equation (none are shrunk to exactly 0).


# 15. Conclusion
* Ridge Regression stabilizes predictions on correlated features using squared penalty regularization.


# 16. Beginner ML Dictionary

Here are simple, one-sentence explanations of common Machine Learning terms to help you review:

* **Feature**: An input variable or column in your dataset used to make predictions (e.g., BMI studied).
* **Target**: The output variable or label you want the model to predict (e.g., final exam score).
* **Training Data**: The portion of the dataset used to teach the model and find patterns.
* **Testing Data**: The portion of the dataset held back to evaluate how well the model performs on new, unseen data.
* **Prediction**: The output value generated by the trained model when given new input features.
* **Overfitting**: A scenario where the model learns the training data too well, including its noise, and performs poorly on new data.
* **Underfitting**: A scenario where the model is too simple to learn the underlying patterns in the training data, leading to poor performance on both training and test data.
* **Model**: The mathematical representation of the patterns learned from the training data by the algorithm.
* **Algorithm**: The set of rules or mathematical procedures followed to build the model from the data (e.g., Linear Regression).
* **Accuracy**: The percentage of correct predictions made by a classification model.
* **Cluster**: A group of similar data points grouped together by an unsupervised learning algorithm based on their characteristics.
* **Centroid**: The center point of a cluster, representing the average location of all data points belonging to that cluster.
